<div style="background:#0e0f11;padding:2.5rem 2rem 2rem;border-radius:16px;margin-bottom:1rem;font-family:'Segoe UI',sans-serif">
  <div style="display:flex;align-items:center;gap:1rem;margin-bottom:1rem">
    <div style="width:44px;height:44px;background:#22c975;border-radius:12px;display:flex;align-items:center;justify-content:center;font-size:1.4rem">🎵</div>
    <div>
      <h1 style="color:white;font-size:1.6rem;margin:0;font-weight:600;letter-spacing:-0.02em">Lectio</h1>
      <p style="color:rgba(255,255,255,.45);margin:0;font-size:.85rem">Free lecture transcription · powered by Whisper AI</p>
    </div>
  </div>
  <p style="color:rgba(255,255,255,.6);margin:0;font-size:.9rem;line-height:1.6">Welcome! This tool transcribes your lecture recordings into text for free.<br><b style='color:white'>Just press the ▶ button below (or Runtime → Run all) and follow the steps.</b></p>
</div>


In [ ]:
#@title ⚙️ Step 1 — Setup (runs automatically, ~2 min first time)

import subprocess, sys, os

# Pretty status display
from IPython.display import display, HTML, clear_output
import ipywidgets as widgets

def show_status(message, color="#1a6b45", bg="#d6efe3", icon="✓"):
    display(HTML(f"""
    <div style="background:{bg};border-radius:10px;padding:.85rem 1.25rem;
                font-family:'Segoe UI',sans-serif;font-size:.9rem;
                color:{color};display:flex;align-items:center;gap:.6rem;margin:.25rem 0">
        <span style="font-size:1rem">{icon}</span> {message}
    </div>
    """))

def show_header(title, subtitle=""):
    display(HTML(f"""
    <div style="background:#0e0f11;border-radius:12px;padding:1.25rem 1.5rem;margin:.75rem 0;
                font-family:'Segoe UI',sans-serif">
        <p style="color:white;margin:0;font-weight:600;font-size:1rem">{title}</p>
        {f'<p style="color:rgba(255,255,255,.45);margin:.25rem 0 0;font-size:.82rem">{subtitle}</p>' if subtitle else ''}
    </div>
    """))

show_header("⚙️ Setting up...", "Installing Whisper AI and tools — this takes ~2 minutes the first time")

# Install quietly
show_status("Installing ffmpeg...", icon="⏳", color="#c47d0e", bg="#fef3dc")
result = subprocess.run(["apt-get", "install", "-y", "-q", "ffmpeg"],
                         capture_output=True, text=True)

show_status("Installing faster-whisper...", icon="⏳", color="#c47d0e", bg="#fef3dc")
result = subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                          "faster-whisper", "tqdm"],
                         capture_output=True, text=True)

# Check GPU
import torch
gpu_available = torch.cuda.is_available()
if gpu_available:
    gpu_name = torch.cuda.get_device_name(0)
    show_status(f"GPU ready: {gpu_name} — transcription will be fast! 🚀")
else:
    show_status("No GPU found — running on CPU (slower). Go to Runtime → Change runtime type → GPU to speed things up.",
                color="#92330a", bg="#fce8e8", icon="⚠️")

show_status("Setup complete! Scroll down to Step 2 👇")


In [ ]:
#@title 🎬 Step 2 — Upload your file & transcribe

from IPython.display import display, HTML, clear_output
import ipywidgets as widgets
from google.colab import files
from faster_whisper import WhisperModel
import torch, os, time

# ── Language selector ────────────────────────────────────────────────────────
display(HTML("""
<div style="font-family:'Segoe UI',sans-serif;margin:.75rem 0">
  <div style="background:#0e0f11;border-radius:12px;padding:1.25rem 1.5rem;margin-bottom:1rem">
    <p style="color:white;margin:0;font-weight:600;font-size:1rem">🌍 Language (optional)</p>
    <p style="color:rgba(255,255,255,.45);margin:.25rem 0 0;font-size:.82rem">Leave on Auto-detect unless you want to force a specific language</p>
  </div>
</div>
"""))

lang_options = {
    "🔍 Auto-detect (recommended)": None,
    "🇮🇱 Hebrew (עברית)": "he",
    "🇺🇸 English": "en",
    "🇸🇦 Arabic (العربية)": "ar",
    "🇷🇺 Russian (Русский)": "ru",
    "🇫🇷 French (Français)": "fr",
    "🇪🇸 Spanish (Español)": "es",
    "🇩🇪 German (Deutsch)": "de",
    "🇵🇹 Portuguese": "pt",
    "🇮🇹 Italian": "it",
    "🇨🇳 Chinese": "zh",
    "🇯🇵 Japanese": "ja",
    "🇹🇷 Turkish": "tr",
    "🇳🇱 Dutch": "nl",
    "🇵🇱 Polish": "pl",
}

lang_dropdown = widgets.Dropdown(
    options=list(lang_options.keys()),
    value="🔍 Auto-detect (recommended)",
    layout=widgets.Layout(width="320px", height="36px")
)
display(lang_dropdown)

# ── Upload ───────────────────────────────────────────────────────────────────
display(HTML("""
<div style="font-family:'Segoe UI',sans-serif;margin:1.25rem 0 .5rem">
  <div style="background:#0e0f11;border-radius:12px;padding:1.25rem 1.5rem;margin-bottom:.75rem">
    <p style="color:white;margin:0;font-weight:600;font-size:1rem">📁 Upload your lecture file</p>
    <p style="color:rgba(255,255,255,.45);margin:.25rem 0 0;font-size:.82rem">MP4, MKV, MOV, AVI, MP3, WAV, OGG, M4A, WEBM — any size</p>
  </div>
</div>
"""))

uploaded = files.upload()

if not uploaded:
    display(HTML('<div style="background:#fce8e8;border-radius:10px;padding:.85rem 1.25rem;color:#92330a;font-family:Segoe UI,sans-serif">⚠️ No file selected. Please run this cell again and choose a file.</div>'))
else:
    filename = list(uploaded.keys())[0]
    filesize_mb = os.path.getsize(filename) / (1024 * 1024)

    display(HTML(f"""
    <div style="background:#d6efe3;border-radius:10px;padding:.85rem 1.25rem;
                font-family:'Segoe UI',sans-serif;color:#1a6b45;margin:.5rem 0">
        ✓ File ready: <b>{filename}</b> ({filesize_mb:.1f} MB)
    </div>
    """))

    # ── Load model ───────────────────────────────────────────────────────────
    device = "cuda" if torch.cuda.is_available() else "cpu"
    compute = "float16" if device == "cuda" else "int8"

    display(HTML("""
    <div style="background:#fef3dc;border-radius:10px;padding:.85rem 1.25rem;
                font-family:'Segoe UI',sans-serif;color:#c47d0e;margin:.5rem 0">
        ⏳ Loading Whisper AI model... (~30 seconds first time)
    </div>
    """))

    model = WhisperModel("large-v2", device=device, compute_type=compute)

    # ── Transcribe ───────────────────────────────────────────────────────────
    selected_lang_key = lang_dropdown.value
    language_code = lang_options[selected_lang_key]

    lang_display = selected_lang_key.split(" ", 1)[1] if " " in selected_lang_key else selected_lang_key

    display(HTML(f"""
    <div style="background:#0e0f11;border-radius:12px;padding:1.25rem 1.5rem;
                font-family:'Segoe UI',sans-serif;margin:.75rem 0">
        <p style="color:white;margin:0;font-weight:600">🎙️ Transcribing...</p>
        <p style="color:rgba(255,255,255,.5);margin:.3rem 0 0;font-size:.85rem">
            File: {filename} · Language: {lang_display} · Device: {device.upper()}
        </p>
        <p style="color:rgba(255,255,255,.35);margin:.2rem 0 0;font-size:.78rem">
            A 1-hour lecture takes ~3-6 minutes. Don't close this tab.
        </p>
    </div>
    """))

    start_time = time.time()

    kwargs = {"beam_size": 5}
    if language_code:
        kwargs["language"] = language_code

    segments, info = model.transcribe(filename, **kwargs)

    # Collect with live progress display
    transcript_lines = []
    progress_out = widgets.Output()
    display(progress_out)

    total_duration = info.duration if info.duration else 1
    last_update = 0

    for segment in segments:
        transcript_lines.append(segment.text.strip())
        pct = min(int((segment.end / total_duration) * 100), 99)

        # Update progress bar every 2%
        if pct >= last_update + 2 or pct >= 99:
            last_update = pct
            elapsed = time.time() - start_time
            estimated = (elapsed / max(pct, 1)) * 100
            remaining = max(0, estimated - elapsed)

            with progress_out:
                clear_output(wait=True)
                display(HTML(f"""
                <div style="background:#e8f5e9;border-radius:10px;padding:1rem 1.25rem;
                            font-family:'Segoe UI',sans-serif">
                    <div style="display:flex;justify-content:space-between;margin-bottom:.5rem">
                        <span style="color:#2e7d32;font-weight:500;font-size:.88rem">Transcribing... {pct}%</span>
                        <span style="color:#66bb6a;font-size:.82rem">{int(remaining)}s remaining</span>
                    </div>
                    <div style="background:#c8e6c9;border-radius:4px;height:8px;overflow:hidden">
                        <div style="background:#43a047;width:{pct}%;height:100%;border-radius:4px;
                                    transition:width .3s"></div>
                    </div>
                    <p style="color:#66bb6a;font-size:.78rem;margin:.5rem 0 0">Last: {segment.text.strip()[:80]}{'...' if len(segment.text.strip()) > 80 else ''}</p>
                </div>
                """))

    elapsed_total = time.time() - start_time

    with progress_out:
        clear_output(wait=True)
        display(HTML(f"""
        <div style="background:#d6efe3;border-radius:10px;padding:1rem 1.25rem;
                    font-family:'Segoe UI',sans-serif">
            <div style="background:#a5d6a7;border-radius:4px;height:8px;margin-bottom:.75rem">
                <div style="background:#43a047;width:100%;height:100%;border-radius:4px"></div>
            </div>
            <span style="color:#1a6b45;font-weight:500">✓ Done in {elapsed_total:.0f} seconds!</span>
        </div>
        """))

    full_transcript = "\n".join(transcript_lines)

    # Stats
    word_count = len(full_transcript.split())
    char_count = len(full_transcript)
    detected_lang = info.language if info.language else "unknown"

    display(HTML(f"""
    <div style="background:#0e0f11;border-radius:12px;padding:1.25rem 1.5rem;
                font-family:'Segoe UI',sans-serif;margin:.75rem 0;
                display:grid;grid-template-columns:repeat(3,1fr);gap:1rem">
        <div>
            <p style="color:rgba(255,255,255,.4);font-size:.72rem;margin:0;text-transform:uppercase;letter-spacing:.08em">Words</p>
            <p style="color:white;font-size:1.4rem;font-weight:600;margin:.2rem 0 0">{word_count:,}</p>
        </div>
        <div>
            <p style="color:rgba(255,255,255,.4);font-size:.72rem;margin:0;text-transform:uppercase;letter-spacing:.08em">Characters</p>
            <p style="color:white;font-size:1.4rem;font-weight:600;margin:.2rem 0 0">{char_count:,}</p>
        </div>
        <div>
            <p style="color:rgba(255,255,255,.4);font-size:.72rem;margin:0;text-transform:uppercase;letter-spacing:.08em">Detected language</p>
            <p style="color:#22c975;font-size:1.4rem;font-weight:600;margin:.2rem 0 0">{detected_lang.upper()}</p>
        </div>
    </div>
    """))

    # Preview
    preview = full_transcript[:500] + ("..." if len(full_transcript) > 500 else "")
    display(HTML(f"""
    <div style="background:#f8f9fa;border:1px solid #e5e1d8;border-radius:10px;
                padding:1.25rem;font-family:'Segoe UI',sans-serif;margin:.5rem 0">
        <p style="font-size:.72rem;font-weight:600;text-transform:uppercase;letter-spacing:.08em;
                  color:#888;margin:0 0 .75rem">Preview (first 500 characters)</p>
        <p style="font-size:.88rem;color:#333;line-height:1.7;margin:0;white-space:pre-wrap">{preview}</p>
    </div>
    """))

    # Save transcript
    output_filename = os.path.splitext(filename)[0] + "_transcript.txt"
    with open(output_filename, "w", encoding="utf-8") as f:
        f.write(full_transcript)

    display(HTML(f"""
    <div style="background:#d6efe3;border-radius:10px;padding:.85rem 1.25rem;
                font-family:'Segoe UI',sans-serif;color:#1a6b45;margin:.5rem 0">
        ✓ Transcript saved as <b>{output_filename}</b> — scroll down to download it!
    </div>
    """))

    # Store for next cell
    import builtins
    builtins._lectio_output_file = output_filename
    builtins._lectio_transcript = full_transcript


In [ ]:
#@title 📥 Step 3 — Download your transcript

from IPython.display import display, HTML
from google.colab import files
import builtins

try:
    output_file = builtins._lectio_output_file
    transcript = builtins._lectio_transcript
    word_count = len(transcript.split())

    display(HTML(f"""
    <div style="background:#0e0f11;border-radius:12px;padding:1.5rem;
                font-family:'Segoe UI',sans-serif;margin:.75rem 0;text-align:center">
        <p style="color:white;font-size:1.1rem;font-weight:600;margin:0 0 .5rem">🎉 Your transcript is ready!</p>
        <p style="color:rgba(255,255,255,.5);font-size:.85rem;margin:0 0 1.25rem">
            {word_count:,} words · {output_file}
        </p>
        <p style="color:rgba(255,255,255,.35);font-size:.78rem;margin:0">
            The file will download automatically. If it doesn't, check your browser's download bar.
        </p>
    </div>
    """))

    files.download(output_file)

    display(HTML("""
    <div style="background:#fef3dc;border-radius:10px;padding:1rem 1.25rem;
                font-family:'Segoe UI',sans-serif;color:#c47d0e;margin:.5rem 0">
        <b>What to do with your transcript:</b><br>
        <span style="font-size:.85rem">📋 Paste it into <b>ChatGPT</b> or <b>Claude</b> to get homework help · 
        Upload it to <b>NotebookLM</b> for a study podcast · 
        Open in any text editor for reading</span>
    </div>

    <div style="background:#f3f0e8;border:1px solid #e5e1d8;border-radius:10px;padding:1rem 1.25rem;
                font-family:'Segoe UI',sans-serif;color:#6b7280;font-size:.82rem;margin:.5rem 0">
        💛 Found Lectio useful? <a href='https://ko-fi.com/yogevshaked' 
        style='color:#c47d0e;font-weight:600' target='_blank'>Buy me a coffee</a> — 
        it keeps this tool free for everyone.
    </div>
    """))

except AttributeError:
    display(HTML("""
    <div style="background:#fce8e8;border-radius:10px;padding:.85rem 1.25rem;
                font-family:'Segoe UI',sans-serif;color:#92330a">
        ⚠️ No transcript found yet. Please run Step 2 first.
    </div>
    """))
